# Turkey Business Activity — Live Footfall with YOLO

Goal: measure **how much commercial activity** there is at high-footfall locations in Turkey
(Istanbul squares & bazaars, Konya) from public live cameras, and turn it into business intelligence:

- **How much / when** — footfall time series + peak-hour profile.
- **What is an anomaly** — rolling z-score on footfall (crowd surge / unusual drop).
- **Prolonged stops** — dwell-time tracking: a person or vehicle that lingers in front of the camera.
- **Is it worth opening a business here?** — a simple footfall + dwell + consistency score.

Cameras and access methods are documented in `docs/turkey_cameras.md`.

> **Important — the live-data question.** A notebook cell runs **once** and stops, so it can never be a
> truly "live" app. The real answer is to **decouple collection from display** (Section 7): a
> `collector.py` process samples 24/7 into a database, and a separate dashboard reads it and auto-refreshes.
> This notebook is for *prototyping the analysis*; the collector + Streamlit app make it live.

### Network reality

The IBB streams (`livestream.ibb.gov.tr`) are public, but reachable only from an **open network** —
your own machine, a VM, or a deployed app. Restricted sandboxes (incl. the environment that generated
this repo) block those hosts via an allowlist. So **run this notebook locally**, where the streams resolve.

## 0. Setup

In [ ]:
%pip install -q ultralytics opencv-python-headless yt-dlp pandas numpy matplotlib

In [ ]:
import sys, time, datetime as dt
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# make the shared app/ package importable from the notebook
sys.path.append(str(Path.cwd().parent))
from app.detect_core import load_model, detect_and_count, grab_frame, resolve_youtube, resolve_stream, VEHICLE_NAMES
from app.cameras import CAMERAS, active_cameras, GRID_CAMERAS

DATA_DIR = Path('../data'); DATA_DIR.mkdir(exist_ok=True)
model = load_model('yolov8n.pt')
print('cameras available:', list(active_cameras()))
print('dashboard grid (4 live cameras):', GRID_CAMERAS)

## 1. Pick a camera

Verified high-footfall Turkey commercial cameras live in `app/cameras.py`. Grand Bazaar, Spice Bazaar,
Taksim and Kadikoy are the densest commerce. The four cameras the live dashboard shows side by side
(`GRID_CAMERAS`) are **Konya - Hukumet Meydani**, **Giresun - Gazi Caddesi**, **Otogar Kavsagi** and
**Kadikoy**.

`resolve_stream(cam)` turns any catalog entry into an openable HLS URL regardless of `kind`:

- `hls` — used directly (IBB / tvkur).
- `youtube` — resolved via yt-dlp.
- `skyline` — Giresun: the tokenized `hd-auth.skylinewebcams.com` playlist, scraped from the page.
- `webcamera24` — Otogar Kavsagi: the embedded tvkur/YouTube player on the webcamera24 page.

The skyline and webcamera24 hosts only resolve from an **open network** (and rotate their tokens), so
run this on your own machine; in a restricted sandbox they fail like the IBB hosts.

In [ ]:
CAM_ID = 'kapali_carsi'   # Grand Bazaar. Grid cams: konya_hukumet, giresun_gazi, otogar_kavsagi, kadikoy
cam = CAMERAS[CAM_ID]
stream_url = resolve_stream(cam)   # handles hls / youtube / skyline / webcamera24
print(cam['name'], '->', stream_url)

## 2. Single-frame check

Confirm the stream decodes and YOLO sees the crowd before collecting anything.

In [ ]:
frame = grab_frame(stream_url)
assert frame is not None, 'Could not read a frame. On a restricted network IBB hosts are blocked — run locally.'
print('frame shape:', frame.shape)
print('counts:', detect_and_count(model, frame))

res = model.predict(frame, conf=0.35, classes=[0,1,2,3,5,7], verbose=False)[0]
plt.figure(figsize=(11, 6))
plt.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)); plt.axis('off')
plt.title(cam['name']); plt.show()

## 3. Footfall time series (sparse sampling)

For the **how much / when** question we don't need every frame — one sample every 15-30s is plenty and
is gentle on the server. This is the same logic the collector runs continuously.

In [ ]:
def footfall_series(stream_url, cam_name, interval_s=20, duration_min=5.0):
    rows, t_end = [], time.time() + duration_min * 60
    while time.time() < t_end:
        ts = dt.datetime.now(dt.timezone.utc)
        f = grab_frame(stream_url)
        c = detect_and_count(model, f) if f is not None else {'person': np.nan, 'vehicles': np.nan}
        rows.append({'ts': ts, 'cam': cam_name, 'person': c.get('person'), 'vehicles': c.get('vehicles')})
        print(f"[{ts:%H:%M:%S}] person={c.get('person')} vehicles={c.get('vehicles')}")
        time.sleep(interval_s)
    return pd.DataFrame(rows)

# short demo run; raise duration_min for real collection (or use the collector for 24/7)
df = footfall_series(stream_url, cam['name'], interval_s=20, duration_min=3.0)
df.to_csv(DATA_DIR / f'footfall_{CAM_ID}.csv', index=False)
df.head()

## 4. Anomalies + peak-hour profile

**Anomaly = rolling z-score > 2.5** on the footfall series: a sudden surge (event/promotion/protest) or an
unusual drop (closure/weather). Peak-hour profile tells you *when* the commercial window is.

In [ ]:
def flag_anomalies(s, window=12, z=2.5):
    mu = s.rolling(window, min_periods=4).mean()
    sd = s.rolling(window, min_periods=4).std().replace(0, np.nan)
    return ((s - mu) / sd).abs() > z

df['ts'] = pd.to_datetime(df['ts'])
df['anomaly'] = flag_anomalies(df['person'])

fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].plot(df['ts'], df['person'], marker='o', label='people')
an = df[df['anomaly'] == True]
ax[0].scatter(an['ts'], an['person'], color='red', zorder=5, label='anomaly')
ax[0].set_title('Footfall over time'); ax[0].legend()

df['hour'] = df['ts'].dt.hour
df.groupby('hour')['person'].mean().plot(kind='bar', ax=ax[1])
ax[1].set_title('Avg people by hour (peak-hour profile)')
plt.tight_layout(); plt.show()

## 5. Dwell-time / prolonged stops (tracking)

"How long does a person or vehicle stay in front of the camera?" needs **object tracking** (stable IDs
across frames), which only works on *consecutive* frames — so here we take a short **dense burst**
(a few fps for ~60s) instead of sparse sampling. Ultralytics `model.track()` (ByteTrack) gives each
object an id; we accumulate how many frames each id is seen and how little it moves.

- **Long dwell + low movement** = lingering: window-shopping / a queue / a parked vehicle.
- High share of *lingering* people is a strong **commercial-quality** signal (people stop, not just pass).

In [ ]:
def dwell_analysis(stream_url, seconds=60, target_fps=3, conf=0.35):
    """Dense burst with tracking. Returns per-track dwell seconds + movement."""
    cap = cv2.VideoCapture(stream_url)
    frames_seen = defaultdict(int)
    centroids = defaultdict(list)
    track_cls = {}
    n_frames = int(seconds * target_fps)
    period = 1.0 / target_fps
    try:
        for _ in range(n_frames):
            t0 = time.time()
            ok, frame = cap.read()
            if not ok:
                break
            r = model.track(frame, persist=True, conf=conf, classes=[0,2,3,5,7],
                            tracker='bytetrack.yaml', verbose=False)[0]
            if r.boxes.id is not None:
                for box, tid, cl in zip(r.boxes.xywh.cpu().numpy(),
                                        r.boxes.id.int().cpu().tolist(),
                                        r.boxes.cls.int().cpu().tolist()):
                    frames_seen[tid] += 1
                    centroids[tid].append((float(box[0]), float(box[1])))
                    track_cls[tid] = cl
            time.sleep(max(0, period - (time.time() - t0)))
    finally:
        cap.release()

    from app.detect_core import NAME_BY_ID
    rows = []
    for tid, n in frames_seen.items():
        pts = np.array(centroids[tid])
        movement = float(np.linalg.norm(pts.max(0) - pts.min(0))) if len(pts) > 1 else 0.0
        rows.append({'track_id': tid,
                     'class': NAME_BY_ID.get(track_cls[tid], str(track_cls[tid])),
                     'dwell_s': round(n / target_fps, 1),
                     'movement_px': round(movement, 1)})
    return pd.DataFrame(rows).sort_values('dwell_s', ascending=False)

dwell = dwell_analysis(stream_url, seconds=60, target_fps=3)
dwell.head(15)

In [ ]:
# Flag prolonged stationary objects: long dwell AND little movement.
PERSON_DWELL_S, VEHICLE_DWELL_S, MAX_MOVE_PX = 25, 40, 60
if not dwell.empty:
    is_person = dwell['class'] == 'person'
    stationary = dwell[((is_person & (dwell['dwell_s'] >= PERSON_DWELL_S)) |
                        (~is_person & (dwell['dwell_s'] >= VEHICLE_DWELL_S)))
                       & (dwell['movement_px'] <= MAX_MOVE_PX)]
    print(f"Prolonged stops detected: {len(stationary)}")
    display(stationary)
    linger_rate = (is_person & (dwell['dwell_s'] >= PERSON_DWELL_S)).sum() / max(1, is_person.sum())
    print(f"Linger rate (people who stayed >= {PERSON_DWELL_S}s): {linger_rate:.0%}")

## 5b. Re-identification — "have I seen this person before?"

The detection counts above tell you *how many* people are visible at any moment, but they
double-count anyone who lingers in front of the camera. To answer questions like *"how many
unique customers walked by today?"* or *"is that the same delivery van I saw yesterday?"*
we need **re-identification**: a persistent identity attached to each person/vehicle that
survives across frames, bursts and days.

The implementation is in `app/reid.py`:

1. For each YOLO detection, crop the bounding box.
2. Build a *masked* HSV color histogram (8x8x8 bins, V<30 pixels ignored — kills the
   sodium-yellow night cast on the Konya square) plus aspect ratio + normalized area.
3. L2-normalize -> 514-dim appearance vector.
4. Compare to every entity of the same class already in `data/reid.db` via cosine
   similarity. If the best match is >= `threshold` (default 0.92) we update its
   `sightings` and `last_seen`; otherwise we register a new entity.

This is a **demo-grade signature**. It works well in daylight (different clothing colors
give clearly different histograms). It produces false matches at night when the whole
scene is yellow-tinted — swap `embed_crop()` for an OSNet/torchreid forward pass for
production-grade re-ID; the SQLite registry around it stays the same.

In [ ]:
from app.detect_core import load_model, grab_frame, detect_with_boxes, annotate
from app.reid import ReidStore
import cv2, time
import matplotlib.pyplot as plt

REID_DB = '../data/reid_notebook.db'
Path(REID_DB).parent.mkdir(parents=True, exist_ok=True)
# fresh registry for the notebook demo so re-runs are reproducible
Path(REID_DB).unlink(missing_ok=True)
reid = ReidStore(REID_DB, threshold=0.92)

# use the model we already loaded above; lower conf so we catch the small/distant people
# the Konya wide-angle camera shows.
CAM_ID = 'konya_hukumet'
cam = CAMERAS[CAM_ID]
stream_url = cam['url']
print('feeding re-ID from', cam['name'])

In [ ]:
# Sample N frames every `interval_s` seconds, run YOLO on each, push every detection
# through the re-ID registry. ~2 minutes is enough to see returning IDs appear.
N_SAMPLES, INTERVAL_S, CONF = 20, 6, 0.25

rows = []
for i in range(N_SAMPLES):
    f = grab_frame(stream_url)
    if f is None:
        print(f'[{i:02d}] miss'); time.sleep(INTERVAL_S); continue
    counts, boxes = detect_with_boxes(model, f, conf=CONF)
    results = reid.update_from_frame(CAM_ID, f, boxes)
    new = sum(r.is_new for r in results)
    seen_again = len(results) - new
    rows.append({'sample': i, 'person': counts['person'], 'vehicles': counts['vehicles'],
                 'detections': len(boxes), 'new_ids': new, 'seen_again': seen_again})
    print(f'[{i:02d}] person={counts["person"]} vehicles={counts["vehicles"]} '
          f'-> new={new} seen_again={seen_again}')
    time.sleep(INTERVAL_S)

reid_df = pd.DataFrame(rows)
reid_df

In [ ]:
# Roll-up: how many unique entities did we see? how many came back >=3 times?
stats = reid.stats(CAM_ID)
print('Total unique entities (this camera):', stats['total_unique'])
print('Total sightings:', stats['total_sightings'])
for cls, s in stats['per_class'].items():
    print(f"  {cls:10s}  unique={s['unique']}  sightings={s['total_sightings']}  "
          f"regulars(>=3)={s['regulars']}")

print('\nTop returning entities:')
for r in reid.top_regulars(CAM_ID, n=10):
    print(f"  #{r['entity_id']:4d}  {r['cls']:8s}  sightings={r['sightings']}  "
          f"first={r['first_seen']}  last={r['last_seen']}")

In [ ]:
# Visual: returning-visitor curve — what fraction of detections are 'seen again' over time?
if len(reid_df) >= 3:
    reid_df = reid_df.copy()
    reid_df['returning_rate'] = (reid_df['seen_again'] /
                                 reid_df['detections'].replace(0, np.nan))
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(reid_df['sample'], reid_df['new_ids'], marker='o', label='new IDs')
    ax[0].plot(reid_df['sample'], reid_df['seen_again'], marker='s', label='seen again')
    ax[0].set_title('Re-ID activity per sample')
    ax[0].set_xlabel('sample #'); ax[0].set_ylabel('count'); ax[0].legend()

    ax[1].plot(reid_df['sample'], reid_df['returning_rate'].fillna(0), marker='o',
               color='#36d399')
    ax[1].set_title('Returning-visitor rate (seen_again / detections)')
    ax[1].set_xlabel('sample #'); ax[1].set_ylim(0, 1)
    plt.tight_layout(); plt.show()
else:
    print('Not enough samples for the returning-visitor plot.')

IMPORTANT — re-ID quality depends on the scene.
#
At Konya Hukumet Meydani at night the whole scene is uniform sodium yellow.
Color-histogram re-ID will over-merge IDs there. To validate the *concept*, point
the camera at the daylight Grand Bazaar / Spice Bazaar (different clothing colors)
or set `threshold=0.97` to be very conservative about matches.
#
Production path:
  pip install torchreid
  from torchreid.utils import FeatureExtractor
  extractor = FeatureExtractor(model_name='osnet_ain_x1_0', model_path='', device='cpu')
  def embed_crop(crop, cls): return extractor([crop])[0].cpu().numpy()
Then keep the rest of app/reid.py exactly as-is. The 2,048-dim OSNet embedding
survives lighting changes, pose changes, and partial occlusion much better than
a color histogram.

## 6. "Is it worth opening a business here?" — a simple score

Combine three signals into one 0-100 score. Tune the weights to your business type (a cafe wants high
*linger*; a kiosk wants high *throughput*).

- **Volume** — median footfall (raw demand).
- **Linger** — share of people who stop (engagement / conversion potential).
- **Consistency** — low coefficient of variation (steady traffic beats spiky).

In [ ]:
def business_score(footfall_df, dwell_df, w=(0.5, 0.3, 0.2)):
    people = footfall_df['person'].dropna()
    volume = float(people.median()) if len(people) else 0.0
    cv = float(people.std() / people.mean()) if people.mean() else 1.0
    consistency = max(0.0, 1 - cv)
    is_p = dwell_df['class'] == 'person'
    linger = float((is_p & (dwell_df['dwell_s'] >= 25)).sum() / max(1, is_p.sum())) if len(dwell_df) else 0.0
    vol_norm = min(1.0, volume / 40.0)  # ~40 people/frame treated as 'very busy'; tune per camera FOV
    score = 100 * (w[0]*vol_norm + w[1]*linger + w[2]*consistency)
    return {'volume_median': round(volume,1), 'linger_rate': round(linger,2),
            'consistency': round(consistency,2), 'score_0_100': round(score,1)}

print(cam['name'])
business_score(df, dwell)

## 7. Making it LIVE — collector + dashboard (the real answer)

Everything above is a **one-shot** measurement: the cell runs, then stops. To get an app whose numbers
keep updating, you split the system into two long-lived processes that share a database:

```
  live streams ->  [ collector.py ]  ->  SQLite (data/footfall.db)  ->  [ streamlit_app.py ]  -> you
                   runs forever,                shared store              reads + auto-refresh
                   samples every Ns                                       every 15s
```

**1) Start the collector (open network required) — it never stops:**
```bash
python -m app.collector --db data/footfall.db --interval 20
```
**2) Start the live dashboard in another terminal:**
```bash
pip install streamlit streamlit-autorefresh
streamlit run app/streamlit_app.py
```

Now the data is genuinely live: the collector keeps appending rows 24/7, and the dashboard re-reads the DB
every 15s — no notebook cell is ever re-run. Deploy the collector under `systemd`/Docker/`nohup` to keep it
alive across reboots.

**Cloud upgrade:** swap the SQLite writes in `collector.py` for a hosted Postgres (e.g. Supabase). Then the
collector can run on a small VM/cron and any number of web clients read the same always-fresh data. See
`docs/turkey_cameras.md` for the architecture notes.

## 8. Compare multiple commercial sites

Loop the footfall sampler over several cameras to rank locations by activity — the input to a
site-selection decision.

In [ ]:
# Rank the four dashboard cameras (plus the bazaars) by activity. resolve_stream handles
# every kind, and we skip any id missing from the catalog or without a resolvable URL.
SITES = GRID_CAMERAS + ['kapali_carsi', 'misir_carsisi']
summary = []
for cid in SITES:
    c = CAMERAS.get(cid)
    if not c or not c.get('url'):
        print(f'{cid}: skipped (not in catalog or no url)')
        continue
    try:
        url = resolve_stream(c)
    except Exception as e:
        print(f'{cid}: resolve failed ({e})')
        continue
    sdf = footfall_series(url, c['name'], interval_s=20, duration_min=2.0)  # short demo per site
    summary.append({'site': c['name'], 'median_people': sdf['person'].median(),
                    'max_people': sdf['person'].max()})
pd.DataFrame(summary).sort_values('median_people', ascending=False)